# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pankaj1281/flyrank_ml_internship/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

I inspect CTR, average position, impressions, and days since last update before testing the signals. These fields can have uneven or heavy-tailed distributions, so averages should be interpreted carefully.

In [8]:
signals = [
    "ctr",
    "avg_position",
    "impressions_90d",
    "days_since_last_update"
]

print(df[signals].describe().T)

                          count         mean           std  min   25%     50%  \
ctr                     30000.0     0.510733      3.279162  0.0   0.0    0.07   
avg_position            30000.0    16.342380     15.216790  0.0   6.2   10.80   
impressions_90d         30000.0  5200.366300  16838.019547  1.0  81.0  731.00   
days_since_last_update  30000.0    46.098300     42.078709  1.0  20.0   20.00   

                            75%       max  
ctr                        0.29     100.0  
avg_position              22.30     245.0  
impressions_90d         3615.25  517715.0  
days_since_last_update   104.00     373.0  


In [9]:
for col in signals:
    print(f"\n{col}")
    print("Median:", df[col].median())
    print("90th percentile:", df[col].quantile(0.90))
    print("99th percentile:", df[col].quantile(0.99))


ctr
Median: 0.07
90th percentile: 0.65
99th percentile: 8.33

avg_position
Median: 10.8
90th percentile: 36.8
99th percentile: 69.90099999999984

impressions_90d
Median: 731.0
90th percentile: 12136.400000000009
99th percentile: 73505.82999999987

days_since_last_update
Median: 20.0
90th percentile: 104.0
99th percentile: 106.0


## Signal Test #1 — Freshness

I test whether content freshness is associated with CTR. The result will determine whether freshness is a useful directional signal for the refresh rule.

In [10]:
freshness_test = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean")
      )
)

freshness_test


,n,avg_ctr,avg_impressions
freshness_tier,,,
0-30,20480,0.609021,4199.614062
181+,174,3.693276,1172.448276
31-90,175,0.117543,6506.748571
91-180,9171,0.238367,7486.665140


### Verdict: MIXED

Freshness does not show a consistent relationship with CTR or impressions. The 181+ group has a high average CTR but only 174 rows, so freshness should be combined with other signals.

## Signal Test #2 — Search Position

I test whether better search position is associated with higher CTR. This checks whether CTR and position provide a useful signal for prioritization.

In [11]:
position_test = (
    df.groupby("position_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean")
      )
)

position_test

,n,avg_ctr
position_tier,,
deep,1319,0.150212
page_1,11814,0.652467
page_3_5,7242,0.222484
striking,7304,0.323239
top_3,2321,1.483611


### Verdict: CONFIRMED

Higher search positions generally have higher average CTR, supporting position as a useful directional signal.

## Signal Test #3 — High Impressions and Low CTR

I test whether pages with high impressions but low CTR form a meaningful opportunity group. This directly checks the assumption behind the HIGH_IMPRESSIONS_LOW_CTR reason code.

In [12]:
high_impressions = df["impressions_90d"].quantile(0.75)
low_ctr = df["ctr"].quantile(0.25)

print("High-impression threshold:", high_impressions)
print("Low-CTR threshold:", low_ctr)

opportunity = df[
    (df["impressions_90d"] >= high_impressions) &
    (df["ctr"] <= low_ctr)
]

print("Opportunity rows:", len(opportunity))

print(
    opportunity[
        ["impressions_90d", "ctr", "avg_position"]
    ].describe()
)

High-impression threshold: 3615.25
Low-CTR threshold: 0.0
Opportunity rows: 146
       impressions_90d    ctr  avg_position
count       146.000000  146.0    146.000000
mean       9334.589041    0.0     26.270548
std       18801.513020    0.0     17.472853
min        3633.000000    0.0      1.500000
25%        4365.250000    0.0      9.925000
50%        5248.500000    0.0     23.600000
75%        7377.750000    0.0     39.325000
max      208678.000000    0.0     76.400000


### Verdict: MIXED

High-impression pages with zero CTR do exist, but the low-CTR threshold is 0.0, so this test only identifies pages with no observed clicks. The signal is useful directionally but should be combined with position and other performance signals.

## 3. The flag-linked test

The refresh rule relies partly on content staleness. I therefore test whether pages with longer periods since their last update show a different performance pattern.

This is a directional check of the rule assumption, not evidence that updating a page causes better performance.

In [13]:
flag_test = (
    df.groupby("freshness_tier")
      .agg(
          n=("content_id", "size"),
          avg_ctr=("ctr", "mean"),
          avg_impressions=("impressions_90d", "mean"),
          avg_position=("avg_position", "mean")
      )
      .reset_index()
)

flag_test


,freshness_tier,n,avg_ctr,avg_impressions,avg_position
0,0-30,20480,0.609021,4199.614062,15.685166
1,181+,174,3.693276,1172.448276,11.325862
2,31-90,175,0.117543,6506.748571,16.538286
3,91-180,9171,0.238367,7486.665140,17.901461


### Verdict: MIXED

Older content does not consistently show worse performance: the 181+ group has higher CTR but only 174 rows. Therefore, staleness is a directional signal, not a standalone refresh rule.

## 4. What this means in practice

The audit suggests that search position and CTR are useful directional signals, while freshness alone is less consistent. A content team should therefore combine multiple signals rather than automatically refreshing every older page. These results support prioritization and decision support, not a causal claim about the effect of refreshing content.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.